# Practial 2: Implementing Tabular Q-Learning for Grid World

### Objective of this Practical
 - Implement a tabular Q-learning agent for grid navigation.
 - How agent updates its Q-table thorugh experience 
 - Apply epsilon-greedy action selection to balance exploration and exploitation.
 - observe how a learned Q-table encodes an agent's policy.

### Requriements 
 - #### Environment setup 

```bash
     # Create and activate a virtual environment
     python3 -m venv practical2_env or python -m venv practical2_env
     source practical2_env/bin/activate 
     # Install required libraries after virtual environment activation
     pip install numpy matplotlib 
```

### Background
Q-Learning is a model-free, off-policy reinforcement learning algorithm. The agent learns the value of
taking a particular action in a given state without needing a model of the environment. These values
are stored in a Q-table, where each entry Q(s, a) represents the expected cumulative reward for taking
action a in state s and following the optimal policy thereafter.

#### The Q-table is updated using Bellman equation:

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]
$$

Where:
- $\alpha$ is the learning rate (0 < alpha <= 1), which determines how much the Q-values are updated based on new experiences.
- $\gamma$ is the discount factor (0 <= gamma < 1), which determines the importance of future rewards.
- $r$ is the immediate reward received after taking action a in state s.
- $s'$ is the resulting state after taking action a in state s.
- $\max_{a'} Q(s', a')$ is the best possible Q-value for the next state.

### Grid World Environment
We will use a simple 4×4 grid world. The agent starts at the top-left cell (state 0) and must reach the
goal at the bottom-right cell (state 15). There is one penalty cell ***(state 9)*** that the agent should avoid. The agent can take four actions: up, down, left, and right. Each action has a reward structure.

Grid Layout:
```
+----+----+----+----+
|  0 |  1 |  2 |  3 |
+----+----+----+----+
|  4 |  5 |  6 |  7 |
+----+----+----+----+
|  8 |  9 | 10 | 11 |
+----+----+----+----+
| 12 | 13 | 14 | 15 |  (Goal)
+----+----+----+----+
```
Available Actions: 0 = Up, 1 = Down, 2 = Left, 3 = Right    

In [4]:
# Importing necessary librariesq
import numpy as np
import random

# ---- Environment Setup [PROVIDED] ----
GRID_SIZE = 4
NUM_STATES = GRID_SIZE * GRID_SIZE  # 16 states
NUM_ACTIONS = 4  # Up, Down, Left, Right
GOAL_STATE = 15
PENALTY_STATE = 9

# ---- Q-Table Initialisation [PROVIDED] ----
Q = np.zeros((NUM_STATES, NUM_ACTIONS))

# ---- Hyperparameters [PROVIDED] ----
alpha = 0.1
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
num_episodes = 500

# ---- Helper: Get Next State [PROVIDED] ----
def get_next_state(state, action):
    row, col = divmod(state, GRID_SIZE)
    if action == 0:
        row = max(row - 1, 0)  # Up
    elif action == 1:
        row = min(row + 1, GRID_SIZE - 1)  # Down
    elif action == 2:
        col = max(col - 1, 0)  # Left
    elif action == 3:
        col = min(col + 1, GRID_SIZE - 1)  # Right
    return row * GRID_SIZE + col

# ---- Helper: Get Reward ----
def get_reward(state):
    # [YOUR TASK 1]
    if state == GOAL_STATE:
        return 10
    if state == PENALTY_STATE:
        return -5
    return -0.1

# ---- Action Selection ----
def choose_action(state, epsilon):
    # [YOUR TASK 2]
    if random.random() < epsilon:
        return random.choice([0, 1, 2, 3])
    return int(np.argmax(Q[state]))

# ---- Training Loop ----
for episode in range(num_episodes):
    state = 0
    done = False

    while not done:
        action = choose_action(state, epsilon)
        next_state = get_next_state(state, action)
        reward = get_reward(next_state)

        # [YOUR TASK 3] Q-table update using the Bellman equation
        best_next_q = np.max(Q[next_state])
        td_target = reward + gamma * best_next_q
        td_error = td_target - Q[state, action]
        Q[state, action] = Q[state, action] + alpha * td_error

        state = next_state

        if state == GOAL_STATE:
            done = True

    # [YOUR TASK 4] Decay epsilon after each episode
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print('Training complete.')
print('Learned Q-Table:')
print(np.round(Q, 2))

Training complete.
Learned Q-Table:
[[ 8.67  9.02  8.81  8.03]
 [ 5.04  5.41  5.29  8.88]
 [ 6.81  9.34  5.32  3.98]
 [ 1.16  6.51  3.42  3.02]
 [ 8.58  9.21  8.83  8.71]
 [ 5.26  0.92  5.29  9.34]
 [ 7.21  9.6   7.11  6.37]
 [ 1.69  9.03  5.93  4.37]
 [ 8.63  9.41  8.98  4.31]
 [ 7.04  9.6   7.22  7.88]
 [ 8.37  9.8   3.33  9.1 ]
 [ 4.43  9.93  6.66  7.07]
 [ 9.07  9.34  9.3   9.6 ]
 [ 4.33  9.53  9.28  9.8 ]
 [ 9.5   9.79  9.55 10.  ]
 [ 0.    0.    0.    0.  ]]


# Practical 2: Tabular Q-Learning for Grid World
This notebook implements a tabular Q-learning agent in a 4x4 grid world.
It completes all sections marked **[YOUR TASK]** in the practical scaffold.

In [6]:
# ---- Verification: Greedy path from start ----
state = 0
path = [state]

for _ in range(20):
    action = np.argmax(Q[state])
    state = get_next_state(state, action)
    path.append(state)
    if state == GOAL_STATE:
        break

print('Greedy path:', path)

Greedy path: [0, 4, 8, 12, 13, 14, 15]
